<a href="https://colab.research.google.com/github/khairuladib94/dl-food-research/blob/main/session-9/notebooks/01_group1_bakery_shelf_life_mlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Group 1: Bakery Shelf-Life Prediction

**Tabular regression with a Keras multilayer perceptron**  
Synthetic teaching dataset. Session 9: Group Deep Learning Challenge.

<a href="https://colab.research.google.com/github/khairuladib94/dl-food-research/blob/main/session-9/notebooks/01_group1_bakery_shelf_life_mlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

> **Colab workflow:** Save a copy in Drive, then choose Runtime > Run all. The required teaching dataset downloads automatically; Google Drive is not mounted.



## How to work
1. Run the baseline.
2. Change one main factor in Experiment 1.
3. Change one main factor in Experiment 2.
4. Record evidence for your five-slide presentation.

In [ ]:
from pathlib import Path
import os, time, json
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, f1_score, recall_score, precision_score,
                             confusion_matrix, ConfusionMatrixDisplay)

SEED = 42
keras.utils.set_random_seed(SEED)
# Portable workshop data setup: local Jupyter first, Colab fallback.
DATA_FILE = 'group1_bakery_shelf_life.csv'
LOCAL_DATA_DIR = Path('../data')
if (LOCAL_DATA_DIR / DATA_FILE).exists():
    DATA_DIR = LOCAL_DATA_DIR
    IN_COLAB = False
else:
    from urllib.request import urlretrieve
    try:
        import google.colab  # type: ignore  # noqa: F401
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False
    DATA_DIR = Path('/content/dl-food-research-data') if IN_COLAB else Path.cwd() / '.workshop-data'
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    target = DATA_DIR / DATA_FILE
    if not target.exists():
        url = 'https://raw.githubusercontent.com/khairuladib94/dl-food-research/main' + '/session-9/data/' + DATA_FILE
        print(f'Downloading {DATA_FILE} ...')
        urlretrieve(url, target)
    print(f'Dataset ready: {target}')
FIG_DIR = Path('/content/dl-food-research-figures') if IN_COLAB else Path('../figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)
print('TensorFlow:', tf.__version__, '| Keras:', keras.__version__)


## 1. Load and explore the data

In [ ]:
df = pd.read_csv(DATA_DIR / 'group1_bakery_shelf_life.csv')
display(df.head())
print('Shape:', df.shape)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].hist(df['shelf_life_days'], bins=20, color='#1f77b4', edgecolor='white')
ax[0].set(title='Shelf-life distribution', xlabel='Days', ylabel='Samples')
cor = df.corr(numeric_only=True)['shelf_life_days'].drop('shelf_life_days').sort_values()
cor.plot.barh(ax=ax[1], color=['#c62828' if v < 0 else '#00897b' for v in cor])
ax[1].set(title='Feature correlation with target', xlabel='Correlation')
plt.tight_layout(); plt.savefig(FIG_DIR/'group1_eda.png', dpi=150); plt.show()

## 2. Prepare train, validation and untouched test sets

In [ ]:
X = df.drop(columns='shelf_life_days').values.astype('float32')
y = df['shelf_life_days'].values.astype('float32')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.20, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=.20, random_state=SEED)
scaler = StandardScaler().fit(X_train)
X_train, X_val, X_test = scaler.transform(X_train), scaler.transform(X_val), scaler.transform(X_test)
print(X_train.shape, X_val.shape, X_test.shape)

## 3. Baseline and two controlled experiments

In [ ]:
def build_mlp(units=(8,), dropout=0.0, lr=0.01):
    model = keras.Sequential([keras.layers.Input((X_train.shape[1],))])
    for width in units:
        model.add(keras.layers.Dense(width, activation='relu'))
        if dropout: model.add(keras.layers.Dropout(dropout))
    model.add(keras.layers.Dense(1))
    model.compile(optimizer=keras.optimizers.Adam(lr), loss='mse', metrics=['mae'])
    return model

def run(name, epochs=24, **kwargs):
    keras.utils.set_random_seed(SEED)
    model = build_mlp(**kwargs)
    hist = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=epochs,
                     batch_size=32, verbose=0)
    pred = model.predict(X_test, verbose=0).ravel()
    return model, hist, pred, {'run': name, 'MAE': mean_absolute_error(y_test,pred),
                               'RMSE': mean_squared_error(y_test,pred)**.5, 'R2': r2_score(y_test,pred)}

baseline = run('Baseline: 8 units', units=(8,), lr=.01, epochs=10)
exp1 = run('Exp 1: 32-16 units', units=(32,16), lr=.005)
exp2 = run('Exp 2: 32-16 + dropout', units=(32,16), dropout=.10, lr=.003)
results = pd.DataFrame([baseline[3], exp1[3], exp2[3]])
display(results.round(3))

### YOUR TASK
Modify `units`, `dropout` or `lr`. Change one main factor at a time and add your run to `results`.

In [ ]:
best = min([baseline, exp1, exp2], key=lambda r: r[3]['RMSE'])
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
for run_obj in [baseline, exp1, exp2]:
    ax[0].plot(run_obj[1].history['val_mae'], label=run_obj[3]['run'])
ax[0].set(title='Validation learning curves', xlabel='Epoch', ylabel='MAE'); ax[0].legend(fontsize=7)
ax[1].scatter(y_test, best[2], alpha=.65, color='#1565c0')
lims=[min(y_test.min(),best[2].min()),max(y_test.max(),best[2].max())]
ax[1].plot(lims,lims,'--',color='#c62828'); ax[1].set(title='Best model: predicted vs actual',xlabel='Actual days',ylabel='Predicted days')
plt.tight_layout(); plt.savefig(FIG_DIR/'group1_results.png', dpi=150); plt.show()
print('Presentation-ready summary:', best[3])